# Exhaustive search for trees with exactly three main eigenvalues (SageMath)

This notebook performs the exhaustive computational search over all trees of order $n \leq 35$ for trees with exactly three main eigenvalues.


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import math
import subprocess
import os
import json
import time
import sys
import io
import warnings
from collections import defaultdict
from operator import mul as _mul

# SageMath compatibility: ensure that Python built-in scalar types are used
import builtins
py_int = builtins.int
py_float = builtins.float

# Match Q-all-fix: reserve 2 CPU cores for the system and the notebook by default
CPU_COUNT = os.cpu_count() or 1
NUM_WORKERS = max(1, CPU_COUNT - 2)

%matplotlib inline

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'STHeiti']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print(f"Available CPU cores: {CPU_COUNT}, defaulting to {NUM_WORKERS} worker processes")


In [ ]:
# ===================================================================
# Core functionality: exact integer test for the number of A-main eigenvalues
# ===================================================================
#
# Mathematical principle:
#   The number of A-main eigenvalues equals dim(Krylov subspace K(A, j))
#   Here A is the adjacency matrix and j is the all-ones vector
#
#   Construct W = [j, Aj, A^2 j, A^3 j]  (an n x 4 matrix)
#   The tree has exactly 3 A-main eigenvalues if and only if rank(W) = 3
#
# Adapted optimization (A-Hankel):
#   1. Obtain v1 = Aj = the degree vector directly from the parent array
#   2. Implement A*v by scanning edges, avoiding adjacency-list construction
#   3. The Gram matrix has Hankel structure, so only the 7 parameters h0..h6 are needed
#   4. For trees, h0 = n and h1 = 2(n-1) are constants
#   5. Compute only 5 inner products (h2..h6), then determine the rank via integer determinants
# ===================================================================


def check_a_main_k3_from_parent(parent_array):
    """
    Determine from the parent array whether the tree has exactly 3 A-main eigenvalues.

    Parameters:
        parent_array: list[int], length n with parent[0] = 0,
                      parent[i] (i>=1) is the 1-indexed parent of vertex i

    Returns:
        bool: True if the tree has exactly 3 A-main eigenvalues
    """
    n = len(parent_array)
    if n < 3:
        return False

    # v1 = A*j = degree
    v1 = [0] * n
    for i in range(1, n):
        p = parent_array[i] - 1
        v1[i] += 1
        v1[p] += 1

    # v2 = A*v1, computed by scanning edges directly
    v2 = [0] * n
    for i in range(1, n):
        p = parent_array[i] - 1
        v2[i] += v1[p]
        v2[p] += v1[i]

    # v3 = A*v2
    v3 = [0] * n
    for i in range(1, n):
        p = parent_array[i] - 1
        v3[i] += v2[p]
        v3[p] += v2[i]

    # Gram-Hankel parameters
    h0 = n
    h1 = (n - 1) << 1  # 2*(n-1)
    h2 = sum(map(_mul, v1, v1))
    h3 = sum(map(_mul, v1, v2))
    h4 = sum(map(_mul, v2, v2))
    h5 = sum(map(_mul, v2, v3))
    h6 = sum(map(_mul, v3, v3))

    d4 = (h0 * (h2 * (h4 * h6 - h5 * h5) - h3 * (h3 * h6 - h5 * h4) + h4 * (h3 * h5 - h4 * h4))
        - h1 * (h1 * (h4 * h6 - h5 * h5) - h3 * (h2 * h6 - h5 * h3) + h4 * (h2 * h5 - h4 * h3))
        + h2 * (h1 * (h3 * h6 - h5 * h4) - h2 * (h2 * h6 - h5 * h3) + h4 * (h2 * h4 - h3 * h3))
        - h3 * (h1 * (h3 * h5 - h4 * h4) - h2 * (h2 * h5 - h4 * h3) + h3 * (h2 * h4 - h3 * h3)))

    if d4 != 0:
        return False

    if (h2 * (h4 * h6 - h5 * h5) - h3 * (h3 * h6 - h5 * h4) + h4 * (h3 * h5 - h4 * h4)) != 0:
        return True
    if (h0 * (h4 * h6 - h5 * h5) - h2 * (h2 * h6 - h5 * h3) + h3 * (h2 * h5 - h4 * h3)) != 0:
        return True
    if (h0 * (h2 * h6 - h4 * h4) - h1 * (h1 * h6 - h4 * h3) + h3 * (h1 * h4 - h2 * h3)) != 0:
        return True
    if (h0 * (h2 * h4 - h3 * h3) - h1 * (h1 * h4 - h3 * h2) + h2 * (h1 * h3 - h2 * h2)) != 0:
        return True

    return False


def check_a_main_eigenvalue_count(G, k):
    """
    Compatibility interface: determine whether a networkx graph has exactly k A-main eigenvalues.
    For k = 3, use the fast parent-array path; for other k, use exact rational rank computation.
    """
    n = G.order()
    if n < k:
        return False

    if k == 3:
        nodes = sorted(G.nodes())
        node_map = {v: i for i, v in enumerate(nodes)}
        parent = [0] * n
        from collections import deque
        visited = [False] * n
        visited[0] = True
        queue = deque([0])
        while queue:
            u = queue.popleft()
            for v_orig in G.neighbors(nodes[u]):
                v = node_map[v_orig]
                if not visited[v]:
                    visited[v] = True
                    parent[v] = u + 1
                    queue.append(v)
        return check_a_main_k3_from_parent(parent)

    nodes = list(G.nodes())
    A = nx.to_numpy_array(G, nodelist=nodes, dtype=np.int64)
    j = np.ones(n, dtype=np.int64)
    vecs = [j.copy()]
    cur = j.copy()
    for _ in range(k):
        cur = A @ cur
        vecs.append(cur.copy())

    dim = k + 1
    Gram = [[int(np.dot(vecs[i], vecs[j])) for j in range(dim)] for i in range(dim)]

    import fractions
    mat = [[fractions.Fraction(Gram[i][j]) for j in range(dim)] for i in range(dim)]
    rank = 0
    for col in range(dim):
        pivot = None
        for row in range(rank, dim):
            if mat[row][col] != 0:
                pivot = row
                break
        if pivot is None:
            continue
        mat[rank], mat[pivot] = mat[pivot], mat[rank]
        for row in range(dim):
            if row != rank and mat[row][col] != 0:
                factor = mat[row][col] / mat[rank][col]
                for c in range(dim):
                    mat[row][c] -= factor * mat[rank][c]
        rank += 1

    return rank == k


print("Core decision routine loaded: A-Hankel optimization (edge scan + 5 inner products + exact integer test)")


In [ ]:
# ===================================================================
# Tree generation: use gentreeg -p to output parent arrays
# ===================================================================

def generate_parent_arrays(n, res=None, mod=None):
    """
    Stream all parent arrays of trees of order n.
    """
    n_py = py_int(n)
    cmd = ["gentreeg", "-p", "-q", str(n_py)]
    if res is not None and mod is not None:
        cmd.append(f"{py_int(res)}/{py_int(mod)}")

    try:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=py_int(65536)
        )
    except FileNotFoundError:
        raise RuntimeError("gentreeg was not found. Please make sure nauty/gentreeg is installed and on PATH")

    bad_lines = 0
    line_count = 0

    try:
        for line in proc.stdout:
            line = line.strip()
            if not line:
                continue
            line_count += 1
            try:
                parent = list(map(py_int, line.split()))
            except ValueError:
                bad_lines += 1
                continue
            if len(parent) != n_py:
                bad_lines += 1
                continue
            yield parent

        stderr_output = proc.stderr.read()
        return_code = py_int(proc.wait())

        if return_code != 0:
            raise RuntimeError(
                f"gentreeg failed with return code {return_code}.\nstderr: {stderr_output.strip()}"
            )
        if bad_lines > 0:
            raise RuntimeError(
                f"Enumeration for n={n_py} finished, but parsing failed on {bad_lines}/{line_count} lines!"
            )
    finally:
        for stream in [proc.stdout, proc.stderr]:
            try:
                if stream: stream.close()
            except: pass
        try:
            if proc.poll() is None:
                proc.kill()
                proc.wait()
        except: pass


def parent_array_to_nx_graph(parent_array):
    """Convert a parent array to a networkx graph (used only for plotting)."""
    n = len(parent_array)
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(1, n):
        G.add_edge(i, parent_array[i] - 1)
    return G


def diameter_from_parent(parent_array):
    """Compute the tree diameter directly from the parent array (two BFS passes)."""
    n = len(parent_array)
    if n <= 1:
        return 0
    adj = [[] for _ in range(n)]
    for i in range(1, n):
        p = parent_array[i] - 1
        adj[i].append(p)
        adj[p].append(i)

    def bfs_farthest(start):
        from collections import deque
        dist = [-1] * n
        dist[start] = 0
        queue = deque([start])
        farthest = start
        max_dist = 0
        while queue:
            u = queue.popleft()
            for v in adj[u]:
                if dist[v] == -1:
                    dist[v] = dist[u] + 1
                    if dist[v] > max_dist:
                        max_dist = dist[v]
                        farthest = v
                    queue.append(v)
        return farthest, max_dist

    far1, _ = bfs_farthest(0)
    _, diam = bfs_farthest(far1)
    return diam


print("Tree generator loaded: using parent-array format (no networkx overhead)")

In [ ]:
# ===================================================================
# Parallel analysis with multiprocessing + C worker + Python fallback + atomic saves + integrity checks
# ===================================================================

import tempfile


class _SageJSONEncoder(json.JSONEncoder):
    def default(self, obj):
        try:
            return py_int(obj)
        except (TypeError, ValueError):
            pass
        try:
            return py_float(obj)
        except (TypeError, ValueError):
            pass
        return super().default(obj)


RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath("A_all_mg_fast.ipynb")), "results_main_eigenvalues")


def _atomic_write_json(filepath, data):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    tmp_path = filepath + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=py_int(2), ensure_ascii=False, cls=_SageJSONEncoder)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, filepath)


def _validate_saved_result(data, expected_k=None):
    required = ['n', 'k_value', 'total_trees', 'k3_count', 'elapsed_seconds', 'trees_per_second',
                'k3_trees', 'diameter_distribution', 'timestamp']
    for key in required:
        if key not in data:
            raise ValueError(f"Missing field {key}")

    n = py_int(data['n'])
    if n <= 0:
        raise ValueError("n must be a positive integer")
    if expected_k is not None and py_int(data['k_value']) != py_int(expected_k):
        raise ValueError("k_value mismatch")
    if py_int(data['k3_count']) != len(data['k3_trees']):
        raise ValueError("k3_count does not match the length of k3_trees")
    if py_int(data['total_trees']) < py_int(data['k3_count']):
        raise ValueError("total_trees is smaller than k3_count")

    diam_dist = defaultdict(py_int)
    for tree_info in data['k3_trees']:
        if 'parent_array' not in tree_info or 'diameter' not in tree_info:
            raise ValueError("A k3_trees entry is missing parent_array or diameter")
        parent = tree_info['parent_array']
        if len(parent) != n:
            raise ValueError("parent_array length does not match n")
        d = py_int(tree_info['diameter'])
        diam_dist[str(d)] += 1

    saved_dist = {str(k): py_int(v) for k, v in data['diameter_distribution'].items()}
    if dict(sorted(diam_dist.items())) != dict(sorted(saved_dist.items())):
        raise ValueError("diameter_distribution is inconsistent with k3_trees")


_C_WORKER_SOURCE = r"""
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#define MAXN 64

static int check_k3(const int *pa, int n) {
    if (n < 3) return 0;
    long long v1[MAXN], v2[MAXN], v3[MAXN];
    memset(v1, 0, n * sizeof(long long));
    memset(v2, 0, n * sizeof(long long));
    memset(v3, 0, n * sizeof(long long));

    for (int i = 1; i < n; i++) {
        int p = pa[i] - 1;
        v1[i] += 1;
        v1[p] += 1;
    }
    for (int i = 1; i < n; i++) {
        int p = pa[i] - 1;
        v2[i] += v1[p];
        v2[p] += v1[i];
    }
    for (int i = 1; i < n; i++) {
        int p = pa[i] - 1;
        v3[i] += v2[p];
        v3[p] += v2[i];
    }

    long long h0 = n, h1 = 2LL * (n - 1);
    long long h2 = 0, h3 = 0, h4 = 0, h5 = 0, h6 = 0;
    for (int i = 0; i < n; i++) {
        h2 += v1[i] * v1[i];
        h3 += v1[i] * v2[i];
        h4 += v2[i] * v2[i];
        h5 += v2[i] * v3[i];
        h6 += v3[i] * v3[i];
    }

    typedef __int128 I;
    I d4 = (I)h0 * ((I)h2 * ((I)h4 * h6 - (I)h5 * h5)
                  - (I)h3 * ((I)h3 * h6 - (I)h5 * h4)
                  + (I)h4 * ((I)h3 * h5 - (I)h4 * h4))
         - (I)h1 * ((I)h1 * ((I)h4 * h6 - (I)h5 * h5)
                  - (I)h3 * ((I)h2 * h6 - (I)h5 * h3)
                  + (I)h4 * ((I)h2 * h5 - (I)h4 * h3))
         + (I)h2 * ((I)h1 * ((I)h3 * h6 - (I)h5 * h4)
                  - (I)h2 * ((I)h2 * h6 - (I)h5 * h3)
                  + (I)h4 * ((I)h2 * h4 - (I)h3 * h3))
         - (I)h3 * ((I)h1 * ((I)h3 * h5 - (I)h4 * h4)
                  - (I)h2 * ((I)h2 * h5 - (I)h4 * h3)
                  + (I)h3 * ((I)h2 * h4 - (I)h3 * h3));
    if (d4 != 0) return 0;

    if ((I)h2 * ((I)h4 * h6 - (I)h5 * h5)
      - (I)h3 * ((I)h3 * h6 - (I)h5 * h4)
      + (I)h4 * ((I)h3 * h5 - (I)h4 * h4) != 0) return 1;
    if ((I)h0 * ((I)h4 * h6 - (I)h5 * h5)
      - (I)h2 * ((I)h2 * h6 - (I)h5 * h3)
      + (I)h3 * ((I)h2 * h5 - (I)h4 * h3) != 0) return 1;
    if ((I)h0 * ((I)h2 * h6 - (I)h4 * h4)
      - (I)h1 * ((I)h1 * h6 - (I)h4 * h3)
      + (I)h3 * ((I)h1 * h4 - (I)h2 * h3) != 0) return 1;
    if ((I)h0 * ((I)h2 * h4 - (I)h3 * h3)
      - (I)h1 * ((I)h1 * h4 - (I)h3 * h2)
      + (I)h2 * ((I)h1 * h3 - (I)h2 * h2) != 0) return 1;
    return 0;
}

static int diameter(const int *pa, int n) {
    if (n <= 1) return 0;
    int adj_head[MAXN], adj_next[2 * MAXN], adj_to[2 * MAXN], ec = 0;
    memset(adj_head, -1, n * sizeof(int));
    for (int i = 1; i < n; i++) {
        int p = pa[i] - 1;
        adj_to[ec] = p; adj_next[ec] = adj_head[i]; adj_head[i] = ec++;
        adj_to[ec] = i; adj_next[ec] = adj_head[p]; adj_head[p] = ec++;
    }

    int queue[MAXN], dist[MAXN], far, mx, head, tail;
    memset(dist, -1, n * sizeof(int));
    dist[0] = 0; queue[0] = 0; head = 0; tail = 1; far = 0; mx = 0;
    while (head < tail) {
        int u = queue[head++];
        for (int e = adj_head[u]; e != -1; e = adj_next[e]) {
            int v = adj_to[e];
            if (dist[v] == -1) {
                dist[v] = dist[u] + 1;
                if (dist[v] > mx) { mx = dist[v]; far = v; }
                queue[tail++] = v;
            }
        }
    }

    memset(dist, -1, n * sizeof(int));
    dist[far] = 0; queue[0] = far; head = 0; tail = 1; mx = 0;
    while (head < tail) {
        int u = queue[head++];
        for (int e = adj_head[u]; e != -1; e = adj_next[e]) {
            int v = adj_to[e];
            if (dist[v] == -1) {
                dist[v] = dist[u] + 1;
                if (dist[v] > mx) mx = dist[v];
                queue[tail++] = v;
            }
        }
    }
    return mx;
}

static int parse_parent_line(char *line, int *pa, int n) {
    int parsed = 0;
    char *p = line;
    while (parsed < n && *p) {
        while (*p == ' ' || *p == '\t') p++;
        if (!*p || *p == '\n') break;
        pa[parsed++] = atoi(p);
        while (*p && *p != ' ' && *p != '\t' && *p != '\n') p++;
    }
    return parsed == n;
}

int main(int argc, char **argv) {
    if (argc != 4) {
        fprintf(stderr, "Usage: %s n res mod\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]), res = atoi(argv[2]), mod = atoi(argv[3]);
    if (n > MAXN) {
        fprintf(stderr, "n=%d > MAXN=%d\n", n, MAXN);
        return 1;
    }

    char cmd[256];
    snprintf(cmd, sizeof(cmd), "gentreeg -p -q %d %d/%d", n, res, mod);
    FILE *fp = popen(cmd, "r");
    if (!fp) {
        fprintf(stderr, "popen failed\n");
        return 1;
    }

    char line[4096];
    int pa[MAXN];
    long long count = 0;
    int k3_count = 0, cap = 0;
    int *saved_pa = NULL;
    int *saved_d = NULL;

    while (fgets(line, sizeof(line), fp)) {
        if (!parse_parent_line(line, pa, n)) continue;
        count++;
        if (check_k3(pa, n)) {
            if (k3_count == cap) {
                int new_cap = cap ? cap * 2 : 16;
                int *new_pa = (int *)realloc(saved_pa, (size_t)new_cap * (size_t)n * sizeof(int));
                int *new_d = (int *)realloc(saved_d, (size_t)new_cap * sizeof(int));
                if (!new_pa || !new_d) {
                    free(new_pa);
                    free(new_d);
                    free(saved_pa);
                    free(saved_d);
                    pclose(fp);
                    fprintf(stderr, "realloc failed\n");
                    return 1;
                }
                saved_pa = new_pa;
                saved_d = new_d;
                cap = new_cap;
            }
            memcpy(saved_pa + (size_t)k3_count * (size_t)n, pa, (size_t)n * sizeof(int));
            saved_d[k3_count] = diameter(pa, n);
            k3_count++;
        }
    }

    int rc = pclose(fp);
    if (rc != 0) {
        free(saved_pa);
        free(saved_d);
        fprintf(stderr, "gentreeg failed rc=%d\n", rc);
        return 1;
    }

    printf("{\"count\":%lld,\"k3_trees\":[", count);
    for (int i = 0; i < k3_count; i++) {
        if (i > 0) printf(",");
        printf("{\"parent\":[");
        for (int j = 0; j < n; j++) {
            if (j > 0) printf(",");
            printf("%d", saved_pa[(size_t)i * (size_t)n + (size_t)j]);
        }
        printf("],\"diameter\":%d}", saved_d[i]);
    }
    printf("]}\n");

    free(saved_pa);
    free(saved_d);
    return 0;
}
"""


_WORKER_SCRIPT = r"""
import subprocess, sys, json
from collections import deque
from operator import mul

def check_k3(parent):
    n = len(parent)
    if n < 3:
        return False
    v1 = [0] * n
    for i in range(1, n):
        p = parent[i] - 1
        v1[i] += 1
        v1[p] += 1
    v2 = [0] * n
    for i in range(1, n):
        p = parent[i] - 1
        v2[i] += v1[p]
        v2[p] += v1[i]
    v3 = [0] * n
    for i in range(1, n):
        p = parent[i] - 1
        v3[i] += v2[p]
        v3[p] += v2[i]
    h0 = n
    h1 = (n - 1) << 1
    h2 = sum(map(mul, v1, v1))
    h3 = sum(map(mul, v1, v2))
    h4 = sum(map(mul, v2, v2))
    h5 = sum(map(mul, v2, v3))
    h6 = sum(map(mul, v3, v3))
    d4 = (h0 * (h2 * (h4 * h6 - h5 * h5) - h3 * (h3 * h6 - h5 * h4) + h4 * (h3 * h5 - h4 * h4))
        - h1 * (h1 * (h4 * h6 - h5 * h5) - h3 * (h2 * h6 - h5 * h3) + h4 * (h2 * h5 - h4 * h3))
        + h2 * (h1 * (h3 * h6 - h5 * h4) - h2 * (h2 * h6 - h5 * h3) + h4 * (h2 * h4 - h3 * h3))
        - h3 * (h1 * (h3 * h5 - h4 * h4) - h2 * (h2 * h5 - h4 * h3) + h3 * (h2 * h4 - h3 * h3)))
    if d4 != 0:
        return False
    if (h2 * (h4 * h6 - h5 * h5) - h3 * (h3 * h6 - h5 * h4) + h4 * (h3 * h5 - h4 * h4)) != 0:
        return True
    if (h0 * (h4 * h6 - h5 * h5) - h2 * (h2 * h6 - h5 * h3) + h3 * (h2 * h5 - h4 * h3)) != 0:
        return True
    if (h0 * (h2 * h6 - h4 * h4) - h1 * (h1 * h6 - h4 * h3) + h3 * (h1 * h4 - h2 * h3)) != 0:
        return True
    if (h0 * (h2 * h4 - h3 * h3) - h1 * (h1 * h4 - h3 * h2) + h2 * (h1 * h3 - h2 * h2)) != 0:
        return True
    return False

def diam(parent):
    n = len(parent)
    if n <= 1:
        return 0
    adj = [[] for _ in range(n)]
    for i in range(1, n):
        p = parent[i] - 1
        adj[i].append(p)
        adj[p].append(i)
    def bfs(start):
        dist = [-1] * n
        dist[start] = 0
        q = deque([start])
        far = start
        mx = 0
        while q:
            u = q.popleft()
            for v in adj[u]:
                if dist[v] == -1:
                    dist[v] = dist[u] + 1
                    if dist[v] > mx:
                        mx = dist[v]
                        far = v
                    q.append(v)
        return far, mx
    f, _ = bfs(0)
    _, d = bfs(f)
    return d

n, res, mod = int(sys.argv[1]), int(sys.argv[2]), int(sys.argv[3])
cmd = ["gentreeg", "-p", "-q", str(n), f"{res}/{mod}"]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=65536)
count = 0
k3_trees = []
bad_lines = 0
for line in proc.stdout:
    line = line.strip()
    if not line:
        continue
    try:
        parent = list(map(int, line.split()))
    except ValueError:
        bad_lines += 1
        continue
    if len(parent) != n:
        bad_lines += 1
        continue
    count += 1
    if check_k3(parent):
        k3_trees.append({"parent": parent, "diameter": diam(parent)})
stderr_out = proc.stderr.read()
rc = proc.wait()
if rc != 0:
    print(f"ERROR: gentreeg failed rc={rc}. stderr: {stderr_out.strip()}", file=sys.stderr)
    sys.exit(1)
if bad_lines > 0:
    print(f"ERROR: {bad_lines} bad lines out of {count + bad_lines}", file=sys.stderr)
    sys.exit(1)
json.dump({"count": count, "k3_trees": k3_trees}, sys.stdout)
"""


def _compile_c_worker():
    c_file = os.path.join(tempfile.gettempdir(), '_a_worker.c')
    bin_file = os.path.join(tempfile.gettempdir(), '_a_worker')

    if os.path.exists(bin_file):
        try:
            result = subprocess.run([bin_file, "4", "0", "1"], capture_output=True, text=True, timeout=10)
            if result.returncode == 0:
                test_data = json.loads(result.stdout)
                if py_int(test_data.get('count', -1)) == 2:
                    return bin_file
        except Exception:
            pass

    with open(c_file, 'w', encoding='utf-8') as f:
        f.write(_C_WORKER_SOURCE)

    try:
        result = subprocess.run(['cc', '-O3', '-o', bin_file, c_file],
                                capture_output=True, text=True, timeout=30)
        if result.returncode == 0:
            print("A-version C worker compiled successfully")
            return bin_file
        print(f"A-version C worker compilation failed: {result.stderr[:200]}")
        return None
    except FileNotFoundError:
        print("No C compiler found; using the Python worker")
        return None
    except Exception as e:
        print(f"A-version C worker compilation raised an exception: {e}")
        return None


_C_WORKER_BIN = _compile_c_worker()


def _run_workers_parallel(n, num_workers):
    use_c = _C_WORKER_BIN is not None

    if not use_c:
        worker_file = os.path.join(tempfile.gettempdir(), '_a_main_worker.py')
        with open(worker_file, 'w', encoding='utf-8') as f:
            f.write(_WORKER_SCRIPT)

    procs = []
    for res in range(py_int(num_workers)):
        if use_c:
            cmd = [_C_WORKER_BIN, str(py_int(n)), str(py_int(res)), str(py_int(num_workers))]
        else:
            cmd = [sys.executable, worker_file, str(py_int(n)), str(py_int(res)), str(py_int(num_workers))]
        procs.append(subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True))

    total = 0
    k3_trees = []
    errors = []

    for i, proc in enumerate(procs):
        stdout, stderr = proc.communicate()
        if proc.returncode != 0:
            errors.append(f"Worker {i} failed (return code {proc.returncode}): {stderr[:300]}")
            continue
        try:
            result = json.loads(stdout)
            total += py_int(result['count'])
            for tree_info in result['k3_trees']:
                k3_trees.append((tree_info['parent'], py_int(tree_info['diameter'])))
        except Exception as e:
            errors.append(f"Worker {i} output parsing failed: {e}")

    if errors:
        raise RuntimeError(
            f"n={n}  had  {len(errors)}  failed workers; the result is incomplete and will not be saved or reused:\n"
            + "\n".join(errors)
        )

    return total, k3_trees


def _save_results_for_n(n, k_value, total_trees, k3_trees, elapsed, results_dir):
    result = {
        'n': py_int(n),
        'k_value': py_int(k_value),
        'total_trees': py_int(total_trees),
        'k3_count': py_int(len(k3_trees)),
        'elapsed_seconds': py_float(round(elapsed, 2)),
        'trees_per_second': py_float(round(total_trees / elapsed, 1)) if elapsed > 0 else 0,
        'k3_trees': [
            {'parent_array': [py_int(x) for x in parent], 'diameter': py_int(d)}
            for parent, d in k3_trees
        ],
        'diameter_distribution': {},
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    for _, d in k3_trees:
        key = str(py_int(d))
        result['diameter_distribution'][key] = py_int(result['diameter_distribution'].get(key, 0) + 1)

    filepath = os.path.join(results_dir, f"n{py_int(n)}_k{py_int(k_value)}.json")
    _validate_saved_result(result, expected_k=k_value)
    _atomic_write_json(filepath, result)
    return filepath


def _load_completed_orders(k_value, results_dir):
    completed = {}
    if not os.path.exists(results_dir):
        return completed

    suffix = f'_k{py_int(k_value)}.json'
    for fname in sorted(os.listdir(results_dir)):
        if not (fname.startswith('n') and fname.endswith(suffix)):
            continue
        path = os.path.join(results_dir, fname)
        try:
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            _validate_saved_result(data, expected_k=k_value)
            completed[py_int(data['n'])] = data
        except Exception as e:
            print(f"Skipping incomplete result file: {path} ({e})")
    return completed


def analyze_single_order(n, k_value, num_workers):
    if py_int(k_value) != 3:
        raise NotImplementedError(f"The optimized path in this notebook currently supports only k=3, not k={k_value}.")

    start_time = time.time()

    if py_int(num_workers) <= 1 or py_int(n) <= 14:
        total = 0
        k3_trees = []
        for parent in generate_parent_arrays(n):
            total += 1
            if check_a_main_k3_from_parent(parent):
                k3_trees.append((parent, diameter_from_parent(parent)))
        elapsed = time.time() - start_time
        return total, k3_trees, elapsed

    total, k3_trees = _run_workers_parallel(n, num_workers)
    elapsed = time.time() - start_time
    return total, k3_trees, elapsed


def analyze_trees_optimized(k_value=3, min_order=6, max_order=30,
                            max_save_per_diameter=100,
                            num_workers=None, results_dir=None):
    if num_workers is None:
        num_workers = NUM_WORKERS
    if results_dir is None:
        results_dir = RESULTS_DIR

    worker_type = "C worker (~same setup as Q-all-fix)" if _C_WORKER_BIN else "Python Hankel worker"

    print(f"{'=' * 70}")
    print(f"Analyzing trees with A-main eigenvalue count k={k_value}")
    print(f"Search range: n={min_order} to n={max_order}")
    print(f"Decision method: A-Hankel exact integer rank")
    print(f"Tree format: parent array (avoiding the sparse6 / networkx hot path)")
    print(f"Parallel mode: {num_workers} {worker_type}")
    print(f"Results directory: {results_dir}")
    print(f"{'=' * 70}\n")

    completed = _load_completed_orders(k_value, results_dir)
    if completed:
        print(f"Detected completed orders: {sorted(completed.keys())}")
        print("These orders will be skipped.\n")

    global_stats = {
        'total_trees': 0,
        'k_trees_by_diameter': defaultdict(py_int),
        'saved_trees_by_diameter': defaultdict(list),
    }

    for n_done, data in sorted(completed.items()):
        if min_order <= n_done <= max_order:
            global_stats['total_trees'] += py_int(data['total_trees'])
            for tree_info in data['k3_trees']:
                d = py_int(tree_info['diameter'])
                global_stats['k_trees_by_diameter'][d] += 1
                if len(global_stats['saved_trees_by_diameter'][d]) < max_save_per_diameter:
                    global_stats['saved_trees_by_diameter'][d].append(
                        (tree_info['parent_array'], d, n_done)
                    )

    for n in range(py_int(min_order), py_int(max_order) + 1):
        print(f"\n{'─' * 70}")

        if n in completed:
            data = completed[n]
            print(f"Order n={n}: saved results already exist; skipping")
            print(f"    - Total trees of this order: {data['total_trees']:,}")
            print(f"    - Satisfying k={k_value}: {data['k3_count']}")
            continue

        print(f"Processing all trees of order n={n}...")

        try:
            total, k3_trees, elapsed = analyze_single_order(n, k_value, num_workers)

            global_stats['total_trees'] += total
            for parent, d in k3_trees:
                global_stats['k_trees_by_diameter'][d] += 1
                if len(global_stats['saved_trees_by_diameter'][d]) < max_save_per_diameter:
                    global_stats['saved_trees_by_diameter'][d].append((parent, d, n))

            filepath = _save_results_for_n(n, k_value, total, k3_trees, elapsed, results_dir)

            print(f"  Completed n={n}")
            print(f"    - Total trees of this order: {py_int(total):,}")
            print(f"    - Satisfying k={k_value}: {py_int(len(k3_trees))}")
            print(f"    - Elapsed time: {py_float(elapsed):.2f} seconds")
            if total > 0:
                speed = py_float(total) / py_float(elapsed) if elapsed > 0 else 0.0
                ratio = py_float(len(k3_trees)) / py_float(total) * 100.0
                print(f"    - Speed: {speed:,.0f} trees/second")
                print(f"    - Matching ratio: {ratio:.6f}%")
            if k3_trees:
                diam_dist = defaultdict(py_int)
                for _, d in k3_trees:
                    diam_dist[py_int(d)] += 1
                print(f"    - Diameter distribution: {dict(sorted(diam_dist.items()))}")
            print(f"    - Saved to: {filepath}")

        except Exception as e:
            print(f"  Error: {e}")
            import traceback
            traceback.print_exc()
            break

    print(f"\n{'=' * 70}")
    print("Analysis completed!")
    print(f"{'=' * 70}")
    print(f"\nDiameter distribution of trees with A-main eigenvalue count k={k_value}:")
    print(f"{'Diameter':>8} | {'Count':>10}")
    print("-" * 30)
    for d in sorted(global_stats['k_trees_by_diameter'].keys()):
        count = global_stats['k_trees_by_diameter'][d]
        print(f"{py_int(d):6d} | {py_int(count):10d}")
    total_k = sum(global_stats['k_trees_by_diameter'].values())
    print(f"\nTotal: {py_int(total_k)} trees with k={k_value}")
    print(f"Total checked: {py_int(global_stats['total_trees']):,} trees\n")

    return global_stats


print("Analysis framework loaded: A-version C worker + Python fallback + atomic saves + resume integrity checks")


In [ ]:
def hierarchy_pos(G, root=None, width=1., vert_gap=0.2, vert_loc=0, xcenter=0.5):
    """Create a hierarchical layout for a tree."""
    if not nx.is_tree(G):
        raise TypeError('Graph must be a tree')
    if root is None and len(G) > 0:
        eccentricity = nx.eccentricity(G)
        center_nodes = [v for v in G.nodes() if eccentricity[v] == min(eccentricity.values())]
        root = center_nodes[0]
    elif root is None:
        root = list(G.nodes())[0]

    def _hierarchy_pos(G, root, width=1., vert_gap=0.2, vert_loc=0, xcenter=0.5,
                       pos=None, parent=None, parsed=None):
        if pos is None:
            pos = {root: (xcenter, vert_loc)}
        else:
            pos[root] = (xcenter, vert_loc)
        if parsed is None:
            parsed = [root]
        else:
            parsed.append(root)
        neighbors = list(G.neighbors(root))
        if parent is not None and parent in neighbors:
            neighbors.remove(parent)
        if len(neighbors) != 0:
            dx = width / len(neighbors)
            nextx = xcenter - width / 2 - dx / 2
            for neighbor in neighbors:
                nextx += dx
                pos = _hierarchy_pos(
                    G, neighbor, width=dx, vert_gap=vert_gap,
                    vert_loc=vert_loc - vert_gap, xcenter=nextx,
                    pos=pos, parent=root, parsed=parsed
                )
        return pos
    return _hierarchy_pos(G, root, width, vert_gap, vert_loc, xcenter)


def plot_single_tree(G, ax, show_labels=False):
    """Plot a single tree."""
    n = G.order()
    if n == 0:
        ax.set_title("Empty graph", fontsize=8)
        ax.axis('off')
        return

    tree_id = G.graph.get('id', 'N/A')
    diameter = G.graph.get('diameter', '?')
    k_value = G.graph.get('k_value', '?')

    title_line1 = f"{tree_id} (n={n}, d={diameter}, k={k_value})"
    title = title_line1

    if diameter == 5:
        try:
            center_nodes = nx.center(G)
            if len(center_nodes) == 2:
                u, v = center_nodes[0], center_nodes[1]
                c1, r1, a = 0, 0, []
                neighbors_u = list(G.neighbors(u))
                neighbors_u.remove(v)
                for n_u in neighbors_u:
                    if G.degree(n_u) == 1: c1 += 1
                    else: r1 += 1; a.append(G.degree(n_u) - 1)
                c2, r2, b = 0, 0, []
                neighbors_v = list(G.neighbors(v))
                neighbors_v.remove(u)
                for n_v in neighbors_v:
                    if G.degree(n_v) == 1: c2 += 1
                    else: r2 += 1; b.append(G.degree(n_v) - 1)
                a.sort(); b.sort()
                title = f"{title_line1}\nr1={r1}, c1={c1}, a={a}\nr2={r2}, c2={c2}, b={b}"
        except: pass

    pos = None
    try:
        pos = hierarchy_pos(G, width=5.0, vert_gap=0.5)
    except:
        try: pos = nx.kamada_kawai_layout(G)
        except: pos = nx.spring_layout(G, k=2.0/np.sqrt(n) if n > 1 else 1, iterations=200, seed=42)

    node_size = max(50, min(300, 500 // n))
    edge_width = max(0.8, min(2.5, 30 / n))
    font_size = max(6, min(10, 100 // n))

    try:
        nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#666666',
                              width=edge_width, alpha=0.6, connectionstyle='arc3,rad=0.1')
    except (TypeError, ValueError):
        nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#666666', width=edge_width, alpha=0.6)

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color='#87CEEB',
                          node_size=node_size, edgecolors='#4682B4', linewidths=2, alpha=0.9)

    if show_labels and n <= 20:
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=font_size,
                               font_weight='bold', font_color='#000000')

    ax.set_title(title, fontsize=10, pad=10, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

    if pos:
        x_values = [coord[0] for coord in pos.values()]
        y_values = [coord[1] for coord in pos.values()]
        x_margin = (max(x_values) - min(x_values)) * 0.1 or 1
        y_margin = (max(y_values) - min(y_values)) * 0.1 or 1
        ax.set_xlim(min(x_values) - x_margin, max(x_values) + x_margin)
        ax.set_ylim(min(y_values) - y_margin, max(y_values) + y_margin)


def plot_trees_by_diameter(trees_dict, k_value, trees_per_page=48):
    """Plot trees grouped by diameter."""
    figures = []
    for diameter in sorted(trees_dict.keys()):
        tree_list = trees_dict[diameter]
        if not tree_list: continue
        if diameter >= 6: tree_list = tree_list[:50]
        total_trees = len(tree_list)
        num_pages = math.ceil(total_trees / trees_per_page)
        print(f"\nPlotting trees with diameter={diameter} (total {total_trees}, {num_pages} page(s))...")
        for page in range(num_pages):
            start_idx = page * trees_per_page
            end_idx = min((page + 1) * trees_per_page, total_trees)
            page_trees = tree_list[start_idx:end_idx]
            num_in_page = len(page_trees)
            cols = min(8, int(np.ceil(np.sqrt(num_in_page * 1.2))))
            rows = math.ceil(num_in_page / cols)
            fig = plt.figure(figsize=(cols * 4, rows * 4))
            gs = fig.add_gridspec(rows, cols, hspace=0.4, wspace=0.3)
            title_suffix = "(showing only the first 50)" if diameter >= 6 and total_trees > 50 else ""
            if num_pages == 1:
                title = f"All trees with main-eigenvalue count k={k_value} and diameter={diameter} (total {total_trees}){title_suffix}"
            else:
                title = f"Trees with main-eigenvalue count k={k_value} and diameter={diameter} (page {page+1}/{num_pages}){title_suffix}"
            fig.suptitle(title, fontsize=16, fontweight='bold', y=0.995)
            for i, G in enumerate(page_trees):
                row = i // cols
                col = i % cols
                ax = fig.add_subplot(gs[row, col])
                plot_single_tree(G, ax, show_labels=False)
            if num_pages == 1:
                filename = f"k{k_value}_diameter{diameter}_trees_all.png"
            else:
                filename = f"k{k_value}_diameter{diameter}_trees_page{page+1}_of_{num_pages}.png"
            fig.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
            print(f"  Saved: {filename}")
            figures.append(fig)
    return figures

In [ ]:
# =======================================================
# Main Program - Optimized Version
# =======================================================

print("\n" + "=" * 70)
print(" " * 10 + "Optimized Analysis Program for A-Main Eigenvalues of Trees")
print("=" * 70)

# ============ Configuration ============
K_VALUE = 3                  # main eigenvalue count
MIN_ORDER = 5               # minimum order (matching the original A_all_mg.ipynb)
MAX_ORDER = 35              # maximum order (matching the original A_all_mg.ipynb)
MAX_SAVE_PER_DIAMETER = 100  # maximum number of trees saved per diameter
TREES_PER_PAGE = 48          # number of trees shown per page
# ==================================

print(f"\nConfiguration:")
print(f"  - main eigenvalue count k: {K_VALUE}")
print(f"  - Search range: {MIN_ORDER} - {MAX_ORDER}")
print(f"  - Decision method: exact integer Gram-matrix rank")
print(f"  - Tree format: parent array")
print(f"  - Number of worker processes: {NUM_WORKERS}")
print(f"  - Max trees saved per diameter: {MAX_SAVE_PER_DIAMETER}")

try:
    stats = analyze_trees_optimized(
        k_value=K_VALUE,
        min_order=MIN_ORDER,
        max_order=MAX_ORDER,
        max_save_per_diameter=MAX_SAVE_PER_DIAMETER,
    )

    # Plotting: convert parent arrays to networkx graphs
    if stats['saved_trees_by_diameter']:
        print("\nStarting tree plots...")
        trees_dict = defaultdict(list)
        for d in sorted(stats['saved_trees_by_diameter'].keys()):
            for pa, diam, n_order in stats['saved_trees_by_diameter'][d]:
                G = parent_array_to_nx_graph(pa)
                G.graph['id'] = f'n{n_order}_d{diam}_k{K_VALUE}_{len(trees_dict[d])+1}'
                G.graph['diameter'] = diam
                G.graph['k_value'] = K_VALUE
                trees_dict[d].append(G)
        figures = plot_trees_by_diameter(trees_dict, K_VALUE, TREES_PER_PAGE)
        print(f"\n{'='*70}")
        print(f"Plotting completed! Generated {len(figures)} figure(s)")
        print(f"{'='*70}\n")
    else:
        print(f"\nNo trees with main eigenvalue count {K_VALUE} were found")

except KeyboardInterrupt:
    print("\n\nExecution interrupted by the user (completed orders were saved automatically; restart to resume)")
except Exception as e:
    print(f"\n\nProgram error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 70)
print("Program finished successfully!")
print("=" * 70 + "\n")